## STEP 1: Exploratory Data Analysis:


### Import libraries:

In [2]:
import os
import pandas as pd
import numpy as np 

import json
import cv2
import matplotlib as plt

from PIL import Image

### Load data

In [3]:
print(os.listdir("../data"))

splits = ["test","val","train"]
    

['instances_test2019.json', 'instances_train2019.json', 'instances_val2019.json', 'test2019', 'train2019', 'val2019']


In [4]:
# Now create a variable to store each path

paths = {} #empty dictionary to store paths

for split in splits:
    #Directory PATH:
    paths[f"{split}_path"] = os.path.join('../data/', f"{split}2019")

    #JSON PATHS
    paths[f"{split}_json"] = os.path.join('../data/',f"instances_{split}2019")

#print the dictionary:
print(paths)

{'test_path': '../data/test2019', 'test_json': '../data/instances_test2019', 'val_path': '../data/val2019', 'val_json': '../data/instances_val2019', 'train_path': '../data/train2019', 'train_json': '../data/instances_train2019'}


In [5]:
import os

splits = ["train", "val", "test"]
data_root = "../data"
data_map = {}

for split in splits:
    #sub-dictionary for every split
    data_map[split] = {
        "images": os.path.join(data_root, f"{split}2019"),
        "annotations": os.path.join(data_root, f"instances_{split}2019.json")
    }

# --- Verification Loop ---
for split, info in data_map.items():
    print("-" * 30)
    print(f"Checking Split: {split.upper()}")
    
    img_dir = info["images"]
    ann_file = info["annotations"]

    if os.path.exists(img_dir) and os.path.exists(ann_file):
        num_files = len(os.listdir(img_dir))
        print(f"  [OK] Images Path: {img_dir} ({num_files} files found)")
        print(f"  [OK] JSON Path:   {ann_file}")
    else:
        print(f"  [ERROR] Path mismatch for {split}!")
    print()

------------------------------
Checking Split: TRAIN
  [OK] Images Path: ../data\train2019 (53739 files found)
  [OK] JSON Path:   ../data\instances_train2019.json

------------------------------
Checking Split: VAL
  [OK] Images Path: ../data\val2019 (6000 files found)
  [OK] JSON Path:   ../data\instances_val2019.json

------------------------------
Checking Split: TEST
  [OK] Images Path: ../data\test2019 (24000 files found)
  [OK] JSON Path:   ../data\instances_test2019.json



In [67]:
#Unpack the paths and store them in clear variables for later use:

train_img = data_map["train"]["images"]
train_json = data_map["train"]["annotations"]

val_img = data_map["val"]["images"]
val_json = data_map["val"]["annotations"]

test_img = data_map['test']['images']
test_json = data_map['test']['annotations']

### Explore JSON Files:

In [68]:
#Explore the JSON structure

#Open json file (instance_train.json in this example):
with open(train_json, 'r') as f:
    data = json.load(f)


print(data.keys(), '\n')

if 'images' in data:
    print(f"First Image entries: {data['images'][0]}")

if 'categories' in data:
    print(f"First categories: {data['categories'][0]}")

if 'annotations' in data:   
    print(f"Fisrt annotations entry: {data['annotations'][0]}")

dict_keys(['info', 'licenses', 'categories', '__raw_Chinese_name_df', 'images', 'annotations']) 

First Image entries: {'file_name': '038900004095_camera0-13.jpg', 'width': 2592, 'height': 1944, 'id': 0}
First categories: {'supercategory': 'puffed_food', 'id': 1, 'name': '1_puffed_food'}
Fisrt annotations entry: {'area': 111763.29, 'bbox': [1188.4, 1052.45, 390.96, 285.87], 'category_id': 112, 'id': 0, 'image_id': 0, 'iscrowd': 0, 'segmentation': [[]], 'point_xy': [1383.88, 1195.38]}


In [69]:
#Create a unified dictionary to link each [id] with its corresponding [category]

supercategory_id_to_name = {category['id']:category['supercategory'] for category in data['categories']}
category_id_to_name =  {category['id']:category['name'] for category in data['categories']}

#Confirm the number of categories (should be 200)
print(f"Total number of categories: {len(supercategory_id_to_name)}")
print(f"toatl number of sub_categories: {len(category_id_to_name)}")

print(f"\n Sample category ID: {supercategory_id_to_name.get(8)}")
print(f"Sample category ID: {category_id_to_name.get(8)}")

Total number of categories: 200
toatl number of sub_categories: 200

 Sample category ID: puffed_food
Sample category ID: 8_puffed_food


In [ ]:
#pick random images (to not always pick the first one):

import random
random_img = random.choice(data['images'])
img_id = random_img['id']
file_name = random_img['file_name']

#get image annotation:
img_ann = [ann for ann in data['annotations'] if ann['image_id'] == img_id]

#get the img category / supercategoroy:
img_category = category_id_to_name.get(img_id)
img_supercategory = supercategory_id_to_name.get(img_id)

#print the random image chosen (you should get a random image everytime)

print(random_img) 

### Disiplay Some Images with their bounding Boxes